# Knowledge Graph Memory with Neo4j [Optional]

Flat long-term memory stores facts as **text blobs**. Knowledge graph memory stores them as **entities and relationships** — enabling multi-hop reasoning that vector search struggles with.

```
  (Alice:Customer)-[:PREFERS]->(Email:ContactChannel)
  (Alice:Customer)-[:PLACED]->(ORD-1001:Order)-[:HAS_STATUS]->(Shipped:Status)
  (ORD-1001:Order)-[:SUBJECT_TO]->(ReturnsPolicy:Policy)
```

**Query:** *"Can Alice return her shipped order under policy?"*  
→ Traverse `Alice → ORD-1001 → Shipped` and `ORD-1001 → ReturnsPolicy` — no keyword luck required.

| Approach | Strength | Weakness |
|----------|----------|----------|
| Vector LTM | Fuzzy semantic match | Weak on structured joins |
| **Graph LTM** | Multi-hop, typed relations | Needs schema + extraction |
| RAG corpus | Official policy text | Not per-user facts |

This notebook builds **ShopCo Graph Memory** with:

1. Offline in-memory graph (Neo4j semantics, no server required)
2. Conversation → entity/relation extraction
3. Graph write and multi-hop read for support agents
4. Optional live Neo4j driver section

Marked **[Optional]** — use when your domain has rich entities and relationships.

In [ ]:
"""
ShopCo Support — knowledge graph memory lab (Neo4j optional).
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Literal

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
USE_LIVE_NEO4J = False  # Set True when Neo4j is running locally

NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD", "password")


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


POLICY_SNIPPETS = {
    "returns": "Returns within 30 days. Refunds in 5-7 business days. [pol-returns-01]",
    "shipping": "Free shipping over $50. [pol-shipping-02]",
}

ORDERS_DB = {
    "ORD-1001": {"order_id": "ORD-1001", "status": "shipped", "customer": "alice"},
    "ORD-1002": {"order_id": "ORD-1002", "status": "processing", "customer": "bob"},
}

print("ShopCo graph memory lab ready.")

---

## 1. When Graph Memory Beats Flat Memory

Use knowledge graph memory when:

- Facts connect to **many other facts** (customer → orders → policies)
- You need **deterministic traversal** ("all shipped orders for Alice")
- Relationships have **types** that matter (`PREFERS` vs `ESCALATED_TO`)
- Compliance requires **explainable paths** (audit the graph walk)

Skip graph memory when:

- Only unstructured preferences ("be concise") — vector/LTM is simpler
- Tiny fact count — graph overhead not justified

---

## 2. ShopCo Graph Ontology

In [ ]:
NODE_LABELS = {"Customer", "Order", "Policy", "ContactChannel", "Status", "Product"}

REL_TYPES = {
    "PREFERS",       # Customer → ContactChannel
    "PLACED",        # Customer → Order
    "HAS_STATUS",    # Order → Status
    "SUBJECT_TO",    # Order → Policy
    "PURCHASED",     # Customer → Product
    "SUPERSEDES",    # ContactChannel → ContactChannel (governance)
}

ONTOLOGY_EXAMPLE = """
(alice:Customer {user_id: 'alice', name: 'Alice'})
(alice)-[:PREFERS {since: '2026-07-01'}]->(email:ContactChannel {value: 'email'})
(alice)-[:PLACED]->(ord1001:Order {order_id: 'ORD-1001'})
(ord1001)-[:HAS_STATUS]->(shipped:Status {value: 'shipped'})
(ord1001)-[:SUBJECT_TO]->(returns:Policy {policy_id: 'pol-returns-01'})
"""

print(ONTOLOGY_EXAMPLE)

---

## 3. In-Memory Graph Store — Offline Neo4j Semantics

In [ ]:
@dataclass
class GraphNode:
    id: str
    labels: set[str]
    props: dict[str, Any] = field(default_factory=dict)


@dataclass
class GraphRel:
    id: str
    type: str
    start_id: str
    end_id: str
    props: dict[str, Any] = field(default_factory=dict)


class InMemoryGraphStore:
    """Minimal property graph — mirrors Neo4j node/relationship model."""

    def __init__(self) -> None:
        self.nodes: dict[str, GraphNode] = {}
        self.rels: dict[str, GraphRel] = {}

    def merge_node(self, node_id: str, labels: set[str], props: dict[str, Any]) -> GraphNode:
        if node_id in self.nodes:
            self.nodes[node_id].props.update(props)
            self.nodes[node_id].labels |= labels
            return self.nodes[node_id]
        node = GraphNode(id=node_id, labels=labels, props=props)
        self.nodes[node_id] = node
        return node

    def merge_rel(self, start_id: str, rel_type: str, end_id: str, props: dict[str, Any] | None = None) -> GraphRel:
        for r in self.rels.values():
            if r.start_id == start_id and r.end_id == end_id and r.type == rel_type:
                r.props.update(props or {})
                return r
        rid = f"rel-{uuid.uuid4().hex[:8]}"
        rel = GraphRel(id=rid, type=rel_type, start_id=start_id, end_id=end_id, props=props or {})
        self.rels[rid] = rel
        return rel

    def neighbors(self, node_id: str, rel_type: str | None = None, direction: str = "out") -> list[tuple[GraphNode, GraphRel]]:
        results: list[tuple[GraphNode, GraphRel]] = []
        for rel in self.rels.values():
            if rel_type and rel.type != rel_type:
                continue
            if direction in ("out", "both") and rel.start_id == node_id:
                results.append((self.nodes[rel.end_id], rel))
            if direction in ("in", "both") and rel.end_id == node_id:
                results.append((self.nodes[rel.start_id], rel))
        return results

    def path_find(self, start_id: str, rel_chain: list[str]) -> list[GraphNode]:
        """Follow a typed relationship chain, e.g. ['PLACED', 'HAS_STATUS']."""
        current = start_id
        path = [self.nodes[current]]
        for rt in rel_chain:
            nbrs = self.neighbors(current, rel_type=rt, direction="out")
            if not nbrs:
                return []
            current = nbrs[0][0].id
            path.append(self.nodes[current])
        return path

    def to_cypher_seed(self) -> list[str]:
        """Emit Cypher MERGE statements for optional Neo4j import."""
        stmts: list[str] = []
        for n in self.nodes.values():
            label = list(n.labels)[0] if n.labels else "Node"
            props = ", ".join(f"{k}: {json.dumps(v)}" for k, v in n.props.items())
            stmts.append(f"MERGE (n:{label} {{id: '{n.id}'}}) SET n += {{{props}}}")
        for r in self.rels.values():
            stmts.append(
                f"MATCH (a {{id: '{r.start_id}'}}), (b {{id: '{r.end_id}'}}) "
                f"MERGE (a)-[:{r.type}]->(b)"
            )
        return stmts


GRAPH = InMemoryGraphStore()
print("InMemoryGraphStore ready.")

---

## 4. Cypher Patterns (Reference)

When `USE_LIVE_NEO4J=True`, these are the equivalent Cypher queries:

```cypher
// Merge customer preference
MERGE (c:Customer {user_id: $user_id})
MERGE (ch:ContactChannel {value: $channel})
MERGE (c)-[r:PREFERS]->(ch)
SET r.since = datetime()

// Multi-hop: customer orders and status
MATCH (c:Customer {user_id: $user_id})-[:PLACED]->(o:Order)-[:HAS_STATUS]->(s:Status)
RETURN o.order_id, s.value

// Policy path for an order
MATCH (o:Order {order_id: $order_id})-[:SUBJECT_TO]->(p:Policy)
RETURN p.policy_id, p.summary
```

The offline store implements the same semantics via `merge_node`, `merge_rel`, and `path_find`.

---

## 5. Extract Triples from Conversation

In [ ]:
@dataclass
class Triple:
    subject: str
    subject_label: str
    predicate: str
    obj: str
    object_label: str
    props: dict[str, Any] = field(default_factory=dict)


def extract_triples(user_id: str, message: str) -> list[Triple]:
    """Rule-based entity/relation extraction (offline stand-in for LLM extraction)."""
    triples: list[Triple] = []
    text = message.strip()
    lower = text.lower()

    if "prefer" in lower and "email" in lower:
        triples.append(Triple(user_id, "Customer", "PREFERS", "email", "ContactChannel", {"source": "conversation"}))
    if "prefer" in lower and "phone" in lower:
        triples.append(Triple(user_id, "Customer", "PREFERS", "phone", "ContactChannel", {"source": "conversation"}))

    m = re.search(r"ORD-[0-9]{4}", text.upper())
    if m:
        oid = m.group(0)
        triples.append(Triple(user_id, "Customer", "PLACED", oid, "Order", {"source": "conversation"}))
        row = ORDERS_DB.get(oid)
        if row:
            triples.append(Triple(oid, "Order", "HAS_STATUS", row["status"], "Status", {}))
            triples.append(Triple(oid, "Order", "SUBJECT_TO", "pol-returns-01", "Policy", {}))

    if "return" in lower and not m:
        triples.append(Triple(user_id, "Customer", "ASKED_ABOUT", "returns", "Topic", {}))

    return triples


demo_triples = extract_triples("alice", "My order ORD-1001 is shipped — can I return it?")
for t in demo_triples:
    print(f"({t.subject}:{t.subject_label})-[:{t.predicate}]->({t.obj}:{t.object_label})")

---

## 6. GraphMemoryWriter — Upsert Triples

In [ ]:
class GraphMemoryWriter:
    def __init__(self, graph: InMemoryGraphStore) -> None:
        self.graph = graph

    def _node_id(self, label: str, key: str) -> str:
        return f"{label.lower()}:{key}"

    def write_triple(self, t: Triple) -> None:
        sub_id = self._node_id(t.subject_label, t.subject)
        obj_id = self._node_id(t.object_label, t.obj)
        self.graph.merge_node(sub_id, {t.subject_label}, {"key": t.subject, **t.props})
        self.graph.merge_node(obj_id, {t.object_label}, {"key": t.obj})
        if t.predicate in REL_TYPES:
            self.graph.merge_rel(sub_id, t.predicate, obj_id, {"written_at": utc_now()})

    def write_message(self, user_id: str, message: str) -> list[Triple]:
        triples = extract_triples(user_id, message)
        for t in triples:
            self.write_triple(t)
        return triples

    def supersede_preference(self, user_id: str, old_channel: str, new_channel: str) -> None:
        """Governance: mark old preference superseded in graph."""
        cust = self._node_id("Customer", user_id)
        old_id = self._node_id("ContactChannel", old_channel)
        new_id = self._node_id("ContactChannel", new_channel)
        self.graph.merge_rel(old_id, "SUPERSEDES", new_id, {"at": utc_now()})
        self.write_triple(Triple(user_id, "Customer", "PREFERS", new_channel, "ContactChannel", {"supersedes": old_channel}))


writer = GraphMemoryWriter(GRAPH)
written = writer.write_message("alice", "I prefer email. My order ORD-1001 shipped.")
print(f"Wrote {len(written)} triples, nodes={len(GRAPH.nodes)}, rels={len(GRAPH.rels)}")

---

## 7. GraphMemoryRetriever — Multi-Hop Queries

In [ ]:
class GraphMemoryRetriever:
    def __init__(self, graph: InMemoryGraphStore) -> None:
        self.graph = graph

    def _nid(self, label: str, key: str) -> str:
        return f"{label.lower()}:{key}"

    def customer_preference(self, user_id: str) -> str | None:
        cid = self._nid("Customer", user_id)
        if cid not in self.graph.nodes:
            return None
        for node, _ in self.graph.neighbors(cid, "PREFERS", "out"):
            return str(node.props.get("key", node.id))
        return None

    def customer_orders_with_status(self, user_id: str) -> list[dict[str, str]]:
        cid = self._nid("Customer", user_id)
        results: list[dict[str, str]] = []
        for order_node, _ in self.graph.neighbors(cid, "PLACED", "out"):
            oid = order_node.props.get("key", "")
            status = "unknown"
            for st_node, _ in self.graph.neighbors(order_node.id, "HAS_STATUS", "out"):
                status = str(st_node.props.get("key", ""))
            results.append({"order_id": oid, "status": status})
        return results

    def order_policies(self, order_id: str) -> list[str]:
        oid = self._nid("Order", order_id)
        policies: list[str] = []
        for pol_node, _ in self.graph.neighbors(oid, "SUBJECT_TO", "out"):
            policies.append(str(pol_node.props.get("key", "")))
        return policies

    def explain_path(self, user_id: str, order_id: str) -> str:
        cid = self._nid("Customer", user_id)
        path = self.graph.path_find(cid, ["PLACED", "HAS_STATUS"])
        if not path:
            return f"No graph path from {user_id} through PLACED → HAS_STATUS"
        parts = [f"{n.id} ({','.join(n.labels)})" for n in path]
        pols = self.order_policies(order_id)
        return " → ".join(parts) + (f" | policies: {pols}" if pols else "")


retriever = GraphMemoryRetriever(GRAPH)
print("Preference:", retriever.customer_preference("alice"))
print("Orders:", retriever.customer_orders_with_status("alice"))
print("Path:", retriever.explain_path("alice", "ORD-1001"))

---

## 8. Seed ShopCo Graph

In [ ]:
SHOPCO_GRAPH = InMemoryGraphStore()
shop_writer = GraphMemoryWriter(SHOPCO_GRAPH)

shop_writer.write_message("alice", "I prefer email contact.")
shop_writer.write_message("alice", "Please check order ORD-1001.")
shop_writer.write_message("bob", "My order ORD-1002 is still processing.")

shop_retriever = GraphMemoryRetriever(SHOPCO_GRAPH)
print("Alice orders:", shop_retriever.customer_orders_with_status("alice"))
print("Bob orders:", shop_retriever.customer_orders_with_status("bob"))

---

## 9. ShopCo Graph Support Agent

In [ ]:
POLICY_BY_ID = {
    "pol-returns-01": POLICY_SNIPPETS["returns"],
    "pol-shipping-02": POLICY_SNIPPETS["shipping"],
}


class GraphSupportAgent:
    def __init__(self, writer: GraphMemoryWriter, retriever: GraphMemoryRetriever) -> None:
        self.writer = writer
        self.retriever = retriever

    def handle(self, user_id: str, message: str) -> str:
        self.writer.write_message(user_id, message)
        q = message.lower()

        if "preference" in q or "contact" in q:
            pref = self.retriever.customer_preference(user_id)
            return f"Graph memory shows contact preference: {pref or 'not set'}."

        m = re.search(r"ORD-[0-9]{4}", message.upper())
        if m and ("return" in q or "policy" in q):
            oid = m.group(0)
            pols = self.retriever.order_policies(oid)
            orders = self.retriever.customer_orders_with_status(user_id)
            status = next((o["status"] for o in orders if o["order_id"] == oid), "unknown")
            policy_text = POLICY_BY_ID.get(pols[0], "Policy not found") if pols else "No policy linked"
            path = self.retriever.explain_path(user_id, oid)
            return (
                f"Order {oid} status: {status}.\n"
                f"Policy: {policy_text}\n"
                f"Graph path: {path}"
            )

        if "orders" in q or "my order" in q:
            orders = self.retriever.customer_orders_with_status(user_id)
            if not orders:
                return "No orders in graph memory."
            lines = [f"  {o['order_id']}: {o['status']}" for o in orders]
            return "Your orders (from graph):\n" + "\n".join(lines)

        return "Ask about your orders, preferences, or return policy for an order ID."


agent = GraphSupportAgent(shop_writer, shop_retriever)
print(agent.handle("alice", "Can I return ORD-1001 under policy?"))
print()
print(agent.handle("alice", "What are my contact preferences?"))

---

## 10. Graph vs Vector Memory — Scenario Comparison

In [ ]:
FLAT_MEMORIES = [
    "Alice prefers email contact",
    "Alice placed order ORD-1001",
    "ORD-1001 has status shipped",
    "ORD-1001 subject to pol-returns-01",
]


def flat_search(query: str, memories: list[str]) -> list[str]:
    q_words = set(query.lower().split())
    scored = [(len(q_words & set(m.lower().split())), m) for m in memories]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [m for s, m in scored if s > 0][:2]


QUERY = "return policy for alice shipped order ORD-1001"
flat_hits = flat_search(QUERY, FLAT_MEMORIES)
graph_answer = agent.handle("alice", "Can I return ORD-1001 under policy?")

print("Flat vector/keyword hits (top 2):")
for h in flat_hits:
    print(f"  - {h}")
print("\nGraph agent (multi-hop join):")
print(graph_answer[:300])

---

## 11. Preference Supersede in Graph

In [ ]:
shop_writer.supersede_preference("alice", "email", "phone")
print("Updated preference:", shop_retriever.customer_preference("alice"))

supersede_rels = [r for r in SHOPCO_GRAPH.rels.values() if r.type == "SUPERSEDES"]
print("SUPERSEDES edges:", [(r.start_id, r.end_id) for r in supersede_rels])

---

## 12. Export Cypher for Live Neo4j

In [ ]:
cypher_seed = SHOPCO_GRAPH.to_cypher_seed()
print(f"Generated {len(cypher_seed)} Cypher statements (first 4):")
for stmt in cypher_seed[:4]:
    print(stmt)

---

## 13. Optional Live Neo4j

In [ ]:
def run_live_neo4j_demo() -> None:
    if not USE_LIVE_NEO4J:
        print("Set USE_LIVE_NEO4J = True and ensure Neo4j is running.")
        print(f"Default URI: {NEO4J_URI}")
        return

    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

    def seed(tx: Any) -> None:
        tx.run(
            "MERGE (c:Customer {user_id: $uid}) "
            "MERGE (ch:ContactChannel {value: $ch}) "
            "MERGE (c)-[:PREFERS]->(ch)",
            uid="alice", ch="email",
        )
        tx.run(
            "MERGE (c:Customer {user_id: $uid}) "
            "MERGE (o:Order {order_id: $oid}) "
            "MERGE (c)-[:PLACED]->(o) "
            "MERGE (s:Status {value: $status}) "
            "MERGE (o)-[:HAS_STATUS]->(s)",
            uid="alice", oid="ORD-1001", status="shipped",
        )

    def query_orders(tx: Any, user_id: str) -> list[dict[str, str]]:
        result = tx.run(
            "MATCH (c:Customer {user_id: $uid})-[:PLACED]->(o:Order)-[:HAS_STATUS]->(s:Status) "
            "RETURN o.order_id AS order_id, s.value AS status",
            uid=user_id,
        )
        return [{"order_id": r["order_id"], "status": r["status"]} for r in result]

    with driver.session() as session:
        session.execute_write(seed)
        rows = session.execute_read(query_orders, "alice")
        print("Live Neo4j query result:", rows)
    driver.close()


run_live_neo4j_demo()

---

## 14. Hybrid Architecture — Graph + Vector + Governance

```
  User message
       │
       ├─► Graph writer (structured triples)
       ├─► Vector LTM (unstructured preferences)
       └─► RAG (policy corpus)
       │
       ▼
  Retriever picks path by question type:
    - "my orders"        → graph traversal
    - "be concise"       → vector memory
    - "return policy"    → RAG + graph link to order
```

Graph memory excels at **joins**; vector memory at **fuzzy prose**; RAG at **authoritative docs**. Combine all three in full-stack ShopCo support.

---

## 15. Eval Harness

In [ ]:
EVAL = [
    ("alice preference stored", lambda: shop_retriever.customer_preference("alice") == "phone"),
    ("alice order status", lambda: any(o["order_id"] == "ORD-1001" and o["status"] == "shipped" for o in shop_retriever.customer_orders_with_status("alice"))),
    ("order policy link", lambda: "pol-returns-01" in shop_retriever.order_policies("ORD-1001")),
    ("multi-hop path", lambda: "placed" in shop_retriever.explain_path("alice", "ORD-1001").lower() or "order" in shop_retriever.explain_path("alice", "ORD-1001").lower()),
    ("bob isolated", lambda: shop_retriever.customer_preference("bob") is None),
    ("cypher export", lambda: len(SHOPCO_GRAPH.to_cypher_seed()) >= 5),
]

passed = sum(int(fn()) for _, fn in EVAL)
for label, fn in EVAL:
    print(f"{'✓' if fn() else '✗'} {label}")
print(f"\nEval: {passed}/{len(EVAL)}")

---

## 16. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| No ontology | Inconsistent node labels | Define labels + REL_TYPES upfront |
| Graph for everything | High extraction cost | Hybrid: graph for entities, vector for prose |
| No MERGE / upsert | Duplicate nodes | `merge_node` / `merge_rel` idempotency |
| Skipping governance | Conflicting PREFERS edges | SUPERSEDES + single active preference |
| Neo4j required in dev | Notebook won't run offline | InMemoryGraphStore first |
| LLM extraction without validation | Wrong triples pollute graph | Rule validate + human review |

---

## 17. Production Checklist

- [ ] Domain ontology documented (labels, relationships)
- [ ] Extraction pipeline (rules and/or LLM) with validation
- [ ] Neo4j indexes on `user_id`, `order_id`
- [ ] MERGE-based upserts (idempotent writes)
- [ ] Graph + vector + RAG routing by question type
- [ ] Governance: supersede edges, TTL on episodic nodes
- [ ] Cypher query catalog for support scenarios
- [ ] Offline dev store mirrors production semantics

---

## 18. Optional Live LLM — Triple Extraction

In [ ]:
def extract_triples_llm(user_id: str, message: str) -> list[Triple]:
    if not USE_LIVE_LLM:
        return extract_triples(user_id, message)

    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    prompt = (
        f"Extract ShopCo graph triples from: {message}\n"
        f"Labels: {NODE_LABELS}, Relations: {REL_TYPES}\n"
        f"Return JSON list of {{subject, subject_label, predicate, obj, object_label}}."
    )
    raw = llm.invoke(prompt).content
    try:
        data = json.loads(raw)
        return [Triple(**item) for item in data]
    except (json.JSONDecodeError, TypeError):
        return extract_triples(user_id, message)


print(extract_triples_llm("alice", "ORD-1001 shipped — prefer email")[0])

---

## 19. Quiz

1. When does graph memory outperform flat vector LTM?
2. What does `path_find(['PLACED', 'HAS_STATUS'])` compute?
3. Why use MERGE instead of CREATE in Neo4j?
4. How does `SUPERSEDES` support governance in the graph?
5. Why keep an offline `InMemoryGraphStore` when Neo4j is available?

---

## 20. Summary

**Knowledge graph memory** stores agent LTM as typed entities and relationships:

1. **Ontology** — Customer, Order, Policy, ContactChannel + PREFERS, PLACED, HAS_STATUS
2. **Write** — extract triples from conversation, MERGE into graph
3. **Read** — multi-hop traversal for orders, policies, preferences
4. **Hybrid** — combine graph (joins), vector (prose), RAG (official docs)
5. **Optional Neo4j** — same semantics on `InMemoryGraphStore` or live bolt driver

Use this pattern when ShopCo-style support needs explainable paths like *Customer → Order → Policy* — not just similar text chunks.